## 1. Project Overview

This project is a Mood-Based Anime Recommendation Chatbot built using Python, OpenAI, Gradio, and Jikan API.

The chatbot allows users to:
- choose their current mood
- optionally enter extra preferences
- receive anime recommendations
- view anime posters and trailer links

The goal of this chatbot is to help users quickly discover anime that matches how they feel.

## 1. Import Required Libraries

In [31]:
from dotenv import load_dotenv
import os
import re
import html
import requests
import gradio as gr
from openai import OpenAI

In [32]:
load_dotenv()

True

## 2. Get OPENAI_KEY from .env file

In [33]:
open_ai_key = os.getenv("OPENAI_API_KEY")
print(f"OpenAI API Key: {open_ai_key[:4]}")

OpenAI API Key: sk-p


In [34]:
client = OpenAI(api_key=open_ai_key)

## 3. Build prompt function

In [35]:
def build_prompt(mood: str, extra_preference: str) -> str:
    return f"""
You are a helpful anime recommendation chatbot.

Task:
Recommend exactly 5 anime based on the user's current mood.

Instructions:
- Recommend exactly 5 anime.
- Match the recommendations to the selected mood.
- Consider the extra preference if provided.
- Keep the tone warm, natural, and concise.
- Stay focused only on anime recommendations.
- Prefer safe-for-work anime.
- Return the answer in this exact format:

1. Anime Title
2. Anime Title
3. Anime Title
4. Anime Title
5. Anime Title

User mood: {mood}
Extra preference: {extra_preference.strip() if extra_preference.strip() else "No extra preference"}
"""

In [36]:
build_prompt("happy", "up coming")

"\nYou are a helpful anime recommendation chatbot.\n\nTask:\nRecommend exactly 5 anime based on the user's current mood.\n\nInstructions:\n- Recommend exactly 5 anime.\n- Match the recommendations to the selected mood.\n- Consider the extra preference if provided.\n- Keep the tone warm, natural, and concise.\n- Stay focused only on anime recommendations.\n- Prefer safe-for-work anime.\n- Return the answer in this exact format:\n\n1. Anime Title\n2. Anime Title\n3. Anime Title\n4. Anime Title\n5. Anime Title\n\nUser mood: happy\nExtra preference: up coming\n"

## 4. Get recommendation from LLM

In [37]:
def get_llm_recommendation(mood: str, extra_preference: str) -> str:
    prompt = build_prompt(mood, extra_preference)

    response = client.responses.create(
        model="gpt-4.1",
        input=prompt
    )

    return response.output_text.strip()

In [38]:
get_llm_recommendation("happy", "upcoming")

'1. Kimi ni Todoke: From Me to You (3rd Season – Upcoming July 2024)  \n2. Oshi no Ko (Season 2 – Upcoming July 2024)  \n3. Kengan Ashura Part 3 (Upcoming August 2024)  \n4. My Deer Friend Nokotan (Upcoming July 2024)  \n5. Alya Sometimes Hides Her Feelings in Russian (Upcoming July 2024)'

## 5. Extracting anime titles from Jinkan API results

In [39]:
def extract_anime_titles(llm_text: str):
    titles = []

    for line in llm_text.splitlines():
        line = line.strip()
        if not line:
            continue

        line = re.sub(r"^\d+\.\s*", "", line)

        if line:
            titles.append(line)

    unique_titles = []
    seen = set()
    for t in titles:
        key = t.lower()
        if key not in seen:
            unique_titles.append(t)
            seen.add(key)

    return unique_titles[:5]

In [40]:
extract_anime_titles('1. My Youth Romantic Comedy Is Wrong, As I Expected Season 3  \n2. Blue Period  \n3. Kaguya-sama: Love Is War - Ultra Romantic  \n4. Horimiya  \n5. Assassination Classroom'
)

['My Youth Romantic Comedy Is Wrong, As I Expected Season 3',
 'Blue Period',
 'Kaguya-sama: Love Is War - Ultra Romantic',
 'Horimiya',
 'Assassination Classroom']

## 6. Calling search anime API from Jinkan API

In [41]:
def search_anime_jikan(title: str):
    """
    Search anime with Jikan v4.
    """
    url = "https://api.jikan.moe/v4/anime"
    params = {
        "q": title,
        "limit": 1,
        "sfw": "true"
    }

    response = requests.get(url, params=params, timeout=20)
    response.raise_for_status()

    data = response.json().get("data", [])
    return data[0] if data else None

## 7. Get image from Jinkan API

In [42]:
def get_best_image(anime: dict):
    images = anime.get("images", {})
    jpg = images.get("jpg", {})
    webp = images.get("webp", {})

    return (
        jpg.get("large_image_url")
        or jpg.get("image_url")
        or webp.get("large_image_url")
        or webp.get("image_url")
        or ""
    )

## 8. Extracting trailer url from Jinkan API results

In [43]:
def get_trailer_link(anime: dict):
    trailer = anime.get("trailer") or {}

    if trailer.get("embed_url"):
        youtube_id = trailer["embed_url"].replace("https://www.youtube-nocookie.com/embed/", "").replace("?enablejsapi=1&wmode=opaque&autoplay=1", "")
        return f"https://www.youtube.com/watch?v={youtube_id}"

    return None

## 9. Interacting cards by Jinkan API results

In [44]:
def build_anime_cards(anime_titles):
    cards = []
    for title in anime_titles:
        anime = search_anime_jikan(title)

        if not anime:
            safe_title = html.escape(title)
            cards.append(f"""
            <div style="border:1px solid #ddd; border-radius:14px; padding:16px; margin-bottom:18px;">
                <h3>{safe_title}</h3>
                <p>Anime details not found.</p>
            </div>
            """)
            continue

        anime_title = html.escape(anime.get("title", title))
        year = anime.get("year") or "N/A"
        synopsis = html.escape(anime.get("synopsis") or "No synopsis available.")
        score = anime.get("score") or "N/A"
        episodes = anime.get("episodes") or "N/A"
        image_url = get_best_image(anime)
        trailer_link = get_trailer_link(anime)

        image_html = ""
        if image_url:
            image_html = f"""
            <img src="{image_url}" alt="{anime_title}"
                 style="width:180px; border-radius:12px;" />
            """

        trailer_html = (
            f'<a href="{trailer_link}" target="_blank">Watch trailer</a>'
            if trailer_link else
            "Trailer not found"
        )

        cards.append(f"""
        <div style="border:1px solid #ddd; border-radius:14px; padding:16px; margin-bottom:20px;">
            <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
                <div>{image_html}</div>
                <div style="flex:1; min-width:260px;">
                    <h3 style="margin-top:0;">{anime_title} ({year})</h3>
                    <p><strong>Score:</strong> {score}</p>
                    <p><strong>Episodes:</strong> {episodes}</p>
                    <p>{synopsis}</p>
                    <p>{trailer_html}</p>
                </div>
            </div>
        </div>
        """)

    return "\n".join(cards)

In [45]:
llm_text = get_llm_recommendation('romantic', 'up coming')
print(f'llm_text: ', llm_text)
anime_titles = extract_anime_titles(llm_text)
print(f'anime_titles: ', anime_titles)
anime_cards_html = build_anime_cards(anime_titles)

llm_text:  1. A Sign of Affection  
2. Shikimori’s Not Just a Cutie  
3. My Happy Marriage  
4. The Dangers in My Heart  
5. Insomniacs After School
anime_titles:  ['A Sign of Affection', 'Shikimori’s Not Just a Cutie', 'My Happy Marriage', 'The Dangers in My Heart', 'Insomniacs After School']


## 10. Gradio UI

In [46]:
def select_mood(mood: str):
    return mood, f"### Selected mood: {mood}"

## 11. Submit button's behind logic

In [47]:
def submit_recommendation(selected_mood: str, extra_preference: str):
    if not selected_mood:
        return "Please select a mood first.", ""

    try:
        llm_text = get_llm_recommendation(selected_mood, extra_preference)
        anime_titles = extract_anime_titles(llm_text)
        anime_cards_html = build_anime_cards(anime_titles)
        return llm_text, anime_cards_html
    except Exception as e:
        return f"Error: {str(e)}", ""

## 12. Clear button's behind logic

In [48]:
def clear_all():
    return "", "### Selected mood: None", "", "", ""

## 13. Gradio UI Design

In [49]:
with gr.Blocks() as demo:
    gr.Markdown("# 🌸 Mood-Based Anime Recommendation Chatbot")
    gr.Markdown("Choose your mood, optionally add a preference, then click **Submit**.")

    selected_mood_state = gr.State("")
    selected_mood_display = gr.Markdown("### Selected mood: None")

    extra_preference = gr.Textbox(
        label="Optional preference",
        placeholder="Example: comforting, funny, not too long, slice of life, action"
    )

    gr.Markdown("## Choose your mood")

    with gr.Row():
        happy_btn = gr.Button("😊 Happy", elem_classes="mood-btn")
        sad_btn = gr.Button("😢 Sad", elem_classes="mood-btn")
        stressed_btn = gr.Button("😰 Stressed", elem_classes="mood-btn")
        excited_btn = gr.Button("🤩 Excited", elem_classes="mood-btn")

    with gr.Row():
        romantic_btn = gr.Button("😍 Romantic", elem_classes="mood-btn")
        bored_btn = gr.Button("😴 Bored", elem_classes="mood-btn")
        relaxed_btn = gr.Button("😌 Relaxed", elem_classes="mood-btn")
        inspired_btn = gr.Button("💪 Inspired", elem_classes="mood-btn")

    with gr.Row():
        clear_btn = gr.Button("Clear")
        submit_btn = gr.Button("Submit", elem_classes="submit-btn")

    recommendation_output = gr.Markdown(label="Recommendations")
    anime_output = gr.HTML(label="Anime Results")

    happy_btn.click(fn=lambda: select_mood("Happy"), outputs=[selected_mood_state, selected_mood_display])
    sad_btn.click(fn=lambda: select_mood("Sad"), outputs=[selected_mood_state, selected_mood_display])
    stressed_btn.click(fn=lambda: select_mood("Stressed"), outputs=[selected_mood_state, selected_mood_display])
    excited_btn.click(fn=lambda: select_mood("Excited"), outputs=[selected_mood_state, selected_mood_display])
    romantic_btn.click(fn=lambda: select_mood("Romantic"), outputs=[selected_mood_state, selected_mood_display])
    bored_btn.click(fn=lambda: select_mood("Bored"), outputs=[selected_mood_state, selected_mood_display])
    relaxed_btn.click(fn=lambda: select_mood("Relaxed"), outputs=[selected_mood_state, selected_mood_display])
    inspired_btn.click(fn=lambda: select_mood("Inspired"), outputs=[selected_mood_state, selected_mood_display])

    submit_btn.click(
        fn=submit_recommendation,
        inputs=[selected_mood_state, extra_preference],
        outputs=[recommendation_output, anime_output]
    )

    clear_btn.click(
        fn=clear_all,
        outputs=[selected_mood_state, selected_mood_display, extra_preference, recommendation_output, anime_output]
    )

demo.launch(share=True, inbrowser=True)

* Running on local URL:  http://127.0.0.1:7871


2026/04/01 09:46:26 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-04-01T09:46:26-05:00 is after 2026-04-01T07:01:35Z



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
